# Advanced AI Lab 3 — Image Captioning

This notebook is the **single entry point** for all Lab 3 experiments.  
Run the cells top-to-bottom. All heavy logic lives in the project modules; this notebook only orchestrates them.

| Section | Content |
|---|---|
| 0 — Setup | Install deps, check GPU, verify imports |
| 1 — Theory (Task 3.1) | Pros & cons of each embedding-combination method |
| 2 — Data (Task 3.2) | Load Flickr8k, build vocabulary, create split loaders |
| 3 — Model | Instantiate encoder (ResNet-50) + decoder (LSTM) |
| 4 — Training | Train with per-epoch validation, plot loss curves |
| 5 — Testing | BLEU-1 through BLEU-4 on the held-out test set |
| 6 — Samples | Show images with ground-truth and generated captions |

---
## Section 0 — Setup
Run this cell once before anything else.

In [1]:
import subprocess
import sys

# NLTK punkt tokenizer (needed for BLEU)
import nltk
# Install / upgrade all required packages from requirements.txt
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    check=True,
)

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("Dependencies ready.")


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip


Dependencies ready.


In [2]:
import os
import torch

import config
from data.vocabulary          import Vocabulary
from data.loader              import get_data_loaders, get_split_stats, build_vocabulary_from_train
from data.caption_io          import _load_captions, _split_images
from models.encoder           import ImageEncoder
from models.decoder           import CaptionDecoder
from training.trainer         import train_epoch, validate_epoch, get_criterion
from utils.evaluation         import generate_caption, calculate_bleu
from utils.visualization      import plot_loss_curves, show_sample_captions

# Create output / checkpoint directories
os.makedirs(config.CHECKPOINT_DIR, exist_ok=True)
os.makedirs(config.OUTPUT_DIR,     exist_ok=True)

print("=" * 55)
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    
print(f"Device in use   : {config.DEVICE}")
print("=" * 55)
print(f"Data directory  : {config.DATA_DIR}")
print(f"Checkpoints     : {config.CHECKPOINT_DIR}")
print(f"Outputs         : {config.OUTPUT_DIR}")
print()
print("All project modules loaded successfully.")


PyTorch version : 2.12.0+cu130
CUDA available  : True
GPU             : NVIDIA GeForce RTX 2080 Ti
Device in use   : cuda
Data directory  : /root/Advance_AI/Lab3/dataset/Flickr8k
Checkpoints     : /root/Advance_AI/Lab3/checkpoints
Outputs         : /root/Advance_AI/Lab3/outputs

All project modules loaded successfully.


---
## Section 1 — Theory (Task 3.1)

### Task 3.1.1 — Concatenation for combining embeddings

**Method:** The image feature vector $\mathbf{v}$ and the word embedding $\mathbf{e}_t$ are concatenated along the feature axis at each LSTM time step:
$$\mathbf{x}_t = [\mathbf{v} \;\|\; \mathbf{e}_t] \in \mathbb{R}^{2d}$$

| | Detail |
|---|---|
| **Pros** | ✅ Preserves **all information** from both streams — no information is discarded before the LSTM. |
| | ✅ The LSTM can learn **arbitrary mixing rules** between visual and linguistic signals. |
| | ✅ Conceptually simple and straightforward to implement. |
| | ✅ The two embedding spaces can have **different scales or distributions** without interference (they are not forced into the same space). |
| **Cons** | ❌ **Doubles the input dimensionality**, increasing the number of LSTM parameters (input-to-hidden weight matrix grows). |
| | ❌ With small datasets, the larger parameter count can lead to **overfitting**. |
| | ❌ No explicit interaction between the two modalities before the LSTM — the model must learn cross-modal patterns entirely through the LSTM weights. |

### Task 3.1.2 — Addition for combining embeddings

**Method:** Element-wise sum of the image feature vector and the word embedding (both projected to the same dimension $d$):
$$\mathbf{x}_t = \mathbf{v} + \mathbf{e}_t \in \mathbb{R}^{d}$$

| | Detail |
|---|---|
| **Pros** | ✅ **No dimensional increase** — the LSTM input size stays at $d$, keeping parameter count low. |
| | ✅ Parameter-efficient; works well when data is limited. |
| | ✅ Mathematically equivalent to the **init-add** model studied by Tanti & Gatt — competitive performance at low cost. |
| | ✅ Simple to implement; requires only a single projection layer for alignment. |
| **Cons** | ❌ **Information loss** — if $\mathbf{v}_i = -\mathbf{e}_{t,i}$ for some dimension, the signal cancels to zero. |
| | ❌ **Both embeddings must share the same dimensionality**, which constrains the design space. |
| | ❌ The model cannot distinguish whether a large activation came from vision or language; the mixed signal may be ambiguous. |
| | ❌ Assumes the two embeddings lie in a **compatible semantic space** — projecting a CNN feature into a word-embedding space is a strong prior that may not hold. |

### Task 3.1.3 — Multiplication (element-wise) for combining embeddings

**Method:** Element-wise (Hadamard) product:
$$\mathbf{x}_t = \mathbf{v} \odot \mathbf{e}_t \in \mathbb{R}^{d}$$

| | Detail |
|---|---|
| **Pros** | ✅ **No dimensional increase** — same compact representation as addition. |
| | ✅ Acts as a **gating mechanism**: a near-zero image feature dimension suppresses that word-embedding dimension, giving an implicit attention-like effect. |
| | ✅ Captures **second-order feature interactions** between visual and linguistic components that addition cannot. |
| **Cons** | ❌ **Catastrophic information loss**: if any dimension of $\mathbf{v}$ or $\mathbf{e}_t$ is zero, the entire interaction for that dimension vanishes. |
| | ❌ **Both embeddings must share the same dimension** — same constraint as addition. |
| | ❌ **Gradient flow is fragile** — if either embedding is small, gradients vanish through the multiplication. |
| | ❌ The multiplicative interaction is hard to interpret: the magnitude of the product depends on both modalities simultaneously. |

### Task 3.1.4 — Attention for combining embeddings

**Method:** A learned attention distribution $\boldsymbol{\alpha}_t$ is computed over a set of spatial image feature vectors $\{\mathbf{v}_i\}$, and a context vector $\hat{\mathbf{v}}_t = \sum_i \alpha_{t,i}\,\mathbf{v}_i$ is produced for each time step.  The context and the word embedding are then concatenated or added:
$$\alpha_{t,i} = \text{softmax}(f(\mathbf{h}_{t-1}, \mathbf{v}_i)), \qquad \hat{\mathbf{v}}_t = \sum_i \alpha_{t,i}\,\mathbf{v}_i$$

| | Detail |
|---|---|
| **Pros** | ✅ **Dynamic, context-sensitive weighting** — the model focuses on relevant image regions when generating each word (e.g., "dog" attends to the animal region). |
| | ✅ **Best empirical performance** on standard captioning benchmarks (Xu et al., 2015). |
| | ✅ **Interpretable** — attention maps can be visualised to explain model decisions. |
| | ✅ Handles spatial structure; retains per-location features rather than a single global vector. |
| | ✅ Naturally extends to **Transformer-based** models (self-attention, cross-attention). |
| **Cons** | ❌ **Computationally expensive** — requires computing pairwise scores between each LSTM state and every spatial location at every step. |
| | ❌ Significantly **more complex** to implement correctly (energy function, masking, spatial feature extraction). |
| | ❌ Requires **more training data** to learn meaningful attention patterns. |
| | ❌ Without careful regularisation, attention can collapse to a single location (degenerate solutions). |

### Task 3.1.5 — Difference for combining embeddings

**Method:** Element-wise difference of the two embeddings (both projected to the same dimension):
$$\mathbf{x}_t = \mathbf{v} - \mathbf{e}_t \in \mathbb{R}^{d}$$

| | Detail |
|---|---|
| **Pros** | ✅ **No dimensional increase** — same compact footprint as addition / multiplication. |
| | ✅ Explicitly encodes **contrastive information** — what the image *has* that the current word *lacks* (and vice-versa). |
| | ✅ Useful in tasks where the relationship between modalities is naturally contrastive (e.g., visual question answering about differences). |
| **Cons** | ❌ **Asymmetric and directional** — $\mathbf{v} - \mathbf{e}_t \neq \mathbf{e}_t - \mathbf{v}$, so the representation changes depending on operand order. |
| | ❌ **Significant information loss** — dimensions where the two embeddings are equal cancel to zero, destroying shared signal. |
| | ❌ **Requires same dimensionality** — same constraint as addition and multiplication. |
| | ❌ Rarely used in practice for multimodal fusion; the contrastive signal it provides can usually be captured by concatenation + learned weights at lower cost. |

---
## Section 2 — Data Preparation (Task 3.2)

### Dataset — Flickr8k

Download the Flickr8k dataset from Kaggle: https://www.kaggle.com/datasets/adityajn105/flickr8k

After downloading, place the files so the directory looks like:
```
Lab3/
  dataset/
    Flickr8k/
      Images/          ← all .jpg files
      captions.txt     ← CSV with columns: image, caption
```

The `captions.txt` file should have the format:
```
image,caption
1000268201_693b08cb0e.jpg,A child in a pink dress is climbing up a set of stairs ...
```

In [3]:
# ── Verify dataset files exist ──────────────────────────────────────────── #
from pathlib import Path

assert config.CAPTIONS_FILE.exists(), (
    f"captions.txt not found at {config.CAPTIONS_FILE}.\n"
    "Please download Flickr8k from Kaggle and place it under Lab3/dataset/Flickr8k/"
)
assert config.IMAGES_DIR.exists(), (
    f"Images folder not found at {config.IMAGES_DIR}."
)

print("Dataset files found.")

Dataset files found.


In [4]:
# ── Inspect the split statistics (no DataLoaders yet) ───────────────────── #
stats = get_split_stats()

print("Dataset split summary")
print("=" * 40)
print(f"  Total images    : {stats['total_images']}")
print(f"  Train images    : {stats['train_images']}  ({stats['train_captions']} captions)")
print(f"  Val   images    : {stats['val_images']}   ({stats['val_captions']} captions)")
print(f"  Test  images    : {stats['test_images']}   ({stats['test_captions']} captions)")
print()
print("Overlap verification (all must be 0)")
print(f"  Train ∩ Val  : {stats['overlap_train_val']}")
print(f"  Train ∩ Test : {stats['overlap_train_test']}")
print(f"  Val   ∩ Test : {stats['overlap_val_test']}")

Dataset split summary
  Total images    : 8091
  Train images    : 5663  (28315 captions)
  Val   images    : 1213   (6065 captions)
  Test  images    : 1215   (6075 captions)

Overlap verification (all must be 0)
  Train ∩ Val  : 0
  Train ∩ Test : 0
  Val   ∩ Test : 0


In [5]:
# ── Build vocabulary from TRAINING captions only ─────────────────────────── #
# (Val / test captions must not influence the vocabulary to avoid data leakage)

image_to_captions = _load_captions(config.CAPTIONS_FILE)
all_images        = list(image_to_captions.keys())
train_imgs, val_imgs, test_imgs = _split_images(
    all_images, config.TRAIN_SPLIT, config.VAL_SPLIT, config.SPLIT_SEED
)

vocabulary = build_vocabulary_from_train(
    image_to_captions, train_imgs, min_freq=config.VOCAB_MIN_FREQ
)

print(vocabulary)
print(f"Vocabulary size : {len(vocabulary):,} words")
print(f"Min frequency   : {config.VOCAB_MIN_FREQ}")
print()
# Show a few sample entries
sample_words = list(vocabulary.word2idx.items())[4:14]
print("Sample vocabulary entries (word → idx):")
for w, i in sample_words:
    print(f"  {w!r:20s} → {i}")


Vocabulary(size=2501, min_freq=5)
Vocabulary size : 2,501 words
Min frequency   : 5

Sample vocabulary entries (word → idx):
  '2'                  → 4
  '3'                  → 5
  '4'                  → 6
  '5'                  → 7
  '6'                  → 8
  '8'                  → 9
  'a'                  → 10
  'about'              → 11
  'above'              → 12
  'accordion'          → 13


In [6]:
# ── Build DataLoaders ─────────────────────────────────────────────────────── #
(
    train_loader,
    val_loader,
    test_loader,
    test_ref_loader,
    test_references,
) = get_data_loaders(vocabulary, num_workers=0)

print(f"Train batches : {len(train_loader)}  (batch size {config.BATCH_SIZE})")
print(f"Val   batches : {len(val_loader)}")
print(f"Test  batches : {len(test_loader)}")
print(f"Test  unique images : {len(test_references)}")

# Inspect one batch
sample_imgs, sample_caps = next(iter(train_loader))
print()
print(f"Sample batch — images  : {tuple(sample_imgs.shape)}")
print(f"Sample batch — captions: {tuple(sample_caps.shape)}")
print(f"  Decoded caption[0]   : {vocabulary.decode(sample_caps[0].tolist())}")

Train batches : 885  (batch size 32)
Val   batches : 190
Test  batches : 190
Test  unique images : 1215

Sample batch — images  : (32, 3, 224, 224)
Sample batch — captions: (32, 22)
  Decoded caption[0]   : a smiling girl plays in the street


---
## Section 3 — Model

### Architecture overview

```
Image (3, 224, 224)
    → ImageEncoder (ResNet-50 backbone, frozen)
    → Linear projection + BN + ReLU
    → image_feature  (batch, embed_dim=256)

Caption tokens  [<SOS>, w1, ..., w_{T-1}]
    → Embedding(vocab_size, embed_dim=256)
    → word_embeddings (batch, T, 256)

  Combine (CONCATENATION):
    x_t = [image_feature ∥ word_embedding_t]  →  (batch, T, 512)

    → LSTM(input=512, hidden=512)
    → Dropout
    → Linear(512, vocab_size)
    → logits (batch, T, vocab_size)
```

In [7]:
vocab_size = len(vocabulary)

encoder = ImageEncoder(
    embed_dim=config.EMBED_DIM,
    fine_tune=config.FINE_TUNE_ENCODER,
).to(config.DEVICE)

decoder = CaptionDecoder(
    vocab_size=vocab_size,
    embed_dim=config.EMBED_DIM,
    hidden_dim=config.HIDDEN_DIM,
    num_layers=config.NUM_LAYERS,
    dropout=config.DROPOUT,
).to(config.DEVICE)

# Parameter count
enc_params = sum(p.numel() for p in encoder.parameters() if p.requires_grad)
dec_params = sum(p.numel() for p in decoder.parameters() if p.requires_grad)

print("Encoder")
print(encoder)
print(f"  Trainable parameters : {enc_params:,}")
print()
print("Decoder")
print(decoder)
print(f"  Trainable parameters : {dec_params:,}")
print()
print(f"Total trainable parameters : {enc_params + dec_params:,}")
print(f"Embedding combination      : {config.EMBEDDING_COMBINATION}")

Encoder
ImageEncoder(
  (backbone): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
  

---
## Section 4 — Training

In [8]:
# ── Optimisers ────────────────────────────────────────────────────────────── #
# Only optimise encoder if fine-tuning is enabled
enc_optimizer = (
    torch.optim.Adam(
        filter(lambda p: p.requires_grad, encoder.parameters()),
        lr=config.ENCODER_LR,
    )
    if config.FINE_TUNE_ENCODER else None
)
dec_optimizer = torch.optim.Adam(decoder.parameters(), lr=config.DECODER_LR)

criterion = get_criterion()

print(f"Encoder optimiser : {'Adam (lr={})'.format(config.ENCODER_LR) if enc_optimizer else 'frozen (no optimiser)'}")
print(f"Decoder optimiser : Adam (lr={config.DECODER_LR})")
print(f"Criterion         : CrossEntropyLoss (ignore_index=PAD)")
print(f"Grad clip norm    : {config.GRAD_CLIP}")

Encoder optimiser : Adam (lr=1e-05)
Decoder optimiser : Adam (lr=0.0004)
Criterion         : CrossEntropyLoss (ignore_index=PAD)
Grad clip norm    : 5.0


In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────── #
train_losses = []
val_losses   = []

best_val_loss = float("inf")
CHECKPOINT    = os.path.join(config.CHECKPOINT_DIR, "best_captioner.pt")

# ── wandb init ────────────────────────────────────────────────────────────── #
if config.WANDB_ENABLED:
    import wandb
    wandb.login()           # prompts for API key if not already authenticated
    wandb.init(
        project=config.WANDB_PROJECT,
        name=config.WANDB_RUN_NAME,
        config={
            "embed_dim"         : config.EMBED_DIM,
            "hidden_dim"        : config.HIDDEN_DIM,
            "num_layers"        : config.NUM_LAYERS,
            "dropout"           : config.DROPOUT,
            "batch_size"        : config.BATCH_SIZE,
            "num_epochs"        : config.NUM_EPOCHS,
            "encoder_lr"        : config.ENCODER_LR,
            "decoder_lr"        : config.DECODER_LR,
            "grad_clip"         : config.GRAD_CLIP,
            "fine_tune_encoder" : config.FINE_TUNE_ENCODER,
            "vocab_min_freq"    : config.VOCAB_MIN_FREQ,
            "device"            : str(config.DEVICE),
        },
    )

print(f"Training for {config.NUM_EPOCHS} epochs on {config.DEVICE}")
print("=" * 55)

for epoch in range(1, config.NUM_EPOCHS + 1):
    # ── train ──────────────────────────────────────────────────────────── #
    train_metrics = train_epoch(
        encoder, decoder,
        train_loader, criterion,
        enc_optimizer, dec_optimizer,
        config.DEVICE,
        epoch=epoch,
    )

    # ── validate ───────────────────────────────────────────────────────── #
    val_metrics = validate_epoch(
        encoder, decoder,
        val_loader, criterion,
        config.DEVICE,
        epoch=epoch,
    )

    train_loss = train_metrics["train_loss"]
    val_loss   = val_metrics["val_loss"]

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    # ── checkpoint best model ──────────────────────────────────────────── #
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(
            {
                "epoch"         : epoch,
                "encoder_state" : encoder.state_dict(),
                "decoder_state" : decoder.state_dict(),
                "vocab_size"    : vocab_size,
                "val_loss"      : best_val_loss,
                "config": {
                    "embed_dim"  : config.EMBED_DIM,
                    "hidden_dim" : config.HIDDEN_DIM,
                    "num_layers" : config.NUM_LAYERS,
                },
            },
            CHECKPOINT,
        )
        marker = "  ← best"
        if config.WANDB_ENABLED:
            wandb.run.summary["best_val_loss"] = best_val_loss
            wandb.run.summary["best_epoch"]    = epoch
    else:
        marker = ""

    print(
        f"Epoch [{epoch:02d}/{config.NUM_EPOCHS}]  "
        f"train_loss: {train_loss:.4f}  "
        f"val_loss: {val_loss:.4f}"
        f"{marker}"
    )

print()
print(f"Best val loss : {best_val_loss:.4f}  (checkpoint saved)")

# ── wandb finish ──────────────────────────────────────────────────────────── #
if config.WANDB_ENABLED:
    wandb.finish()


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: deeplab15group (deeplab15group-lule-university-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Training for 30 epochs on cuda


Train:  35%|███▌      | 311/885 [01:27<02:36,  3.66it/s]

In [ ]:
# ── Plot loss curves ─────────────────────────────────────────────────────── #
loss_plot_path = os.path.join(config.OUTPUT_DIR, "loss_curves.png")
plot_loss_curves(train_losses, val_losses, save_path=None)   # inline display
plot_loss_curves(train_losses, val_losses, save_path=loss_plot_path)  # save to disk

---
## Section 5 — Testing: BLEU Score

We load the best checkpoint and evaluate on the **held-out test set**.  
BLEU-4 is the standard metric for image captioning benchmarks.

In [ ]:
# ── Load best checkpoint ─────────────────────────────────────────────────── #
checkpoint = torch.load(CHECKPOINT, map_location=config.DEVICE)
encoder.load_state_dict(checkpoint["encoder_state"])
decoder.load_state_dict(checkpoint["decoder_state"])

print(f"Loaded checkpoint from epoch {checkpoint['epoch']}")
print(f"Checkpoint val loss : {checkpoint['val_loss']:.4f}")

In [ ]:
# ── Compute BLEU scores on the test set ──────────────────────────────────── #
bleu_scores = calculate_bleu(
    encoder, decoder,
    test_ref_loader,
    vocabulary,
    config.DEVICE,
)

print("Test-set BLEU scores")
print("=" * 30)
for metric, score in bleu_scores.items():
    print(f"  {metric} : {score:.4f}")

---
## Section 6 — Sample Caption Generation

Each row shows:  
- **Left**: the test image  
- **Right**: one ground-truth caption and the model's generated caption

In [ ]:
samples_path = os.path.join(config.OUTPUT_DIR, "sample_captions.png")

show_sample_captions(
    encoder, decoder,
    test_ref_loader,
    vocabulary,
    config.DEVICE,
    n=config.NUM_SAMPLE_IMAGES,
    save_path=None,          # display inline
    seed=7,
)

# Also save to disk
show_sample_captions(
    encoder, decoder,
    test_ref_loader,
    vocabulary,
    config.DEVICE,
    n=config.NUM_SAMPLE_IMAGES,
    save_path=samples_path,
    seed=7,
)